# Align Event Catalogs with Time-Series Data

Gravitational-wave detection and instrument glitch studies frequently combine discrete trigger catalogs with continuous auxiliary channel recordings. Catalog timestamps often have timing jitter or coarse arrival estimates that must be refined against high-bandwidth witness channels before extracting aligned multi-channel analysis windows.

**What you will achieve:**
1. Ingest an 8-event catalog and match it with a 3-channel dataset (`WITNESS` [V], `SENSOR` [V], and `DISPLACEMENT` [m]).
2. Refine coarse catalog trigger times to exact local peaks on the witness channel using `find_peaks()`.
3. Construct channel-specific half-open analysis windows using `SegmentTable`.
4. Handle edge boundaries, gaps, and missing detections while preserving table completeness (24 event-channel rows).
5. Align complete events on a common relative time grid and compute robust ensemble statistics.

**Data type**: Synthetic multi-channel time series (600 s at 128 Hz) and discrete event table.

## Setup and Output Directories

In [ ]:
import json
import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy import units as u
from gwpy.segments import Segment, SegmentList

import gwexpy
from gwexpy.table import SegmentTable
from gwexpy.timeseries import TimeSeries, TimeSeriesDict

output_dir = Path(os.environ.get("GWEXPY_DOCS_OUTPUT_DIR") or tempfile.mkdtemp(prefix="gwexpy-t2-"))
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "figures").mkdir(exist_ok=True)
(output_dir / "tables").mkdir(exist_ok=True)
print(f"Output directory: {output_dir}")

## Synthetic Catalog and Multi-Channel Data Contract

- Duration: 600 s, sample rate: 128 Hz ($N = 76800$ samples), $t_0 = 1400000000\,\text{s}$.
- Catalog contains 8 events (`E00` through `E07`) at offsets [1, 60, 120, 180, 240, 360, 480, 599] s.
- Injected timing jitter: [2, -3, 1, 0, -2, 3, -1, 2] samples.
- `E05` has no pulse in witness (negative control).
- `E03` has a sensor gap (NaNs).
- `E00` and `E07` cross data boundaries for long windows.
- Physical channel lags: Sensor delayed by 4 samples (31.25 ms), Displacement delayed by 8 samples (62.5 ms).

In [ ]:
def make_event_fixture(seed: int = 2026091602):
    rng = np.random.default_rng(seed)
    fs = 128.0
    duration = 600.0
    n_samples = int(fs * duration)
    t0_gps = 1400000000.0
    dt = 1.0 / fs

    # Nominal event offsets and injected catalog errors
    design_offsets = np.array([1.0, 60.0, 120.0, 180.0, 240.0, 360.0, 480.0, 599.0])
    jitter_samples = np.array([2, -3, 1, 0, -2, 3, -1, 2])
    catalog_gps = t0_gps + design_offsets + jitter_samples * dt

    catalog_df = pd.DataFrame({
        "event_id": [f"E{i:02d}" for i in range(8)],
        "catalog_gps_s": catalog_gps,
        "design_gps_s": t0_gps + design_offsets
    })

    # Generate signals
    t = np.arange(n_samples) * dt
    data_witness = rng.normal(0, 0.001, n_samples)
    data_sensor = rng.normal(0, 0.0005, n_samples)
    data_disp = rng.normal(0, 1e-12, n_samples)

    pulse_t = np.linspace(-0.15, 0.15, int(0.3 * fs))
    pulse_shape = np.exp(-0.5 * (pulse_t / 0.03)**2)

    for i, offset_s in enumerate(design_offsets):
        center_idx = int(round(offset_s * fs))
        # E05 is negative control: no pulse in witness
        if i != 5:
            p_start = center_idx - len(pulse_shape)//2
            p_end = p_start + len(pulse_shape)
            if 0 <= p_start and p_end <= n_samples:
                data_witness[p_start:p_end] += 1.0 * pulse_shape

        # Sensor has 4 samples delay
        s_center = center_idx + 4
        p_start = s_center - len(pulse_shape)//2
        p_end = p_start + len(pulse_shape)
        if 0 <= p_start and p_end <= n_samples:
            data_sensor[p_start:p_end] += 0.5 * pulse_shape

        # Displacement has 8 samples delay
        d_center = center_idx + 8
        p_start = d_center - len(pulse_shape)//2
        p_end = p_start + len(pulse_shape)
        if 0 <= p_start and p_end <= n_samples:
            data_disp[p_start:p_end] += 1e-9 * pulse_shape

    # Introduce gap near E03 in sensor channel
    gap_center = int(round(180.0 * fs))
    data_sensor[gap_center - 64 : gap_center + 64] = np.nan

    ts_dict = TimeSeriesDict({
        "WITNESS": TimeSeries(data_witness, dt=dt*u.s, t0=t0_gps*u.s, unit=u.V, name="WITNESS"),
        "SENSOR": TimeSeries(data_sensor, dt=dt*u.s, t0=t0_gps*u.s, unit=u.V, name="SENSOR"),
        "DISPLACEMENT": TimeSeries(data_disp, dt=dt*u.s, t0=t0_gps*u.s, unit=u.m, name="DISPLACEMENT"),
    })

    return catalog_df, ts_dict

catalog_df, ts_dict = make_event_fixture()
catalog_df.to_csv(output_dir / "tables/catalog.csv", index=False)
print("Catalog saved to tables/catalog.csv (8 events)")

## Refine Trigger Times with Witness Peaks

Coarse catalog times are refined by searching for local positive peaks in `WITNESS` within a search window of $\pm 0.25\,\text{s}$.

In [ ]:
witness_ts = ts_dict["WITNESS"]
refined_records = []

for _, row in catalog_df.iterrows():
    cat_gps = row["catalog_gps_s"]
    evt_id = row["event_id"]

    # Crop witness around catalog time +- 0.25 s
    w_start = max(cat_gps - 0.25, witness_ts.t0.value)
    w_end = min(cat_gps + 0.25, witness_ts.span[1])

    sub = witness_ts.crop(w_start * u.s, w_end * u.s)
    peaks, props = sub.find_peaks(height=0.5 * u.V, distance=0.1 * u.s)

    if len(peaks) > 0:
        max_idx = np.argmax(props["peak_heights"])
        ref_gps = float(peaks.times.value[max_idx])
        ref_sample = int(round((ref_gps - witness_ts.t0.value) / witness_ts.dt.value))
        cat_sample = int(round((cat_gps - witness_ts.t0.value) / witness_ts.dt.value))
        refined_records.append({
            "event_id": evt_id,
            "catalog_sample": cat_sample,
            "refined_sample": ref_sample,
            "catalog_gps_s": cat_gps,
            "refined_gps_s": ref_gps,
            "offset_s": ref_gps - cat_gps,
            "refine_status": "refined"
        })
    else:
        cat_sample = int(round((cat_gps - witness_ts.t0.value) / witness_ts.dt.value))
        refined_records.append({
            "event_id": evt_id,
            "catalog_sample": cat_sample,
            "refined_sample": cat_sample,
            "catalog_gps_s": cat_gps,
            "refined_gps_s": cat_gps,
            "offset_s": 0.0,
            "refine_status": "no_peak"
        })

refined_df = pd.DataFrame(refined_records)
refined_df.to_csv(output_dir / "tables/refined_events.csv", index=False)
print("Refined events written to tables/refined_events.csv:")
print(refined_df[["event_id", "catalog_gps_s", "refined_gps_s", "refine_status"]])

## Multi-Channel Window Extraction via SegmentTable

We define channel-specific half-open intervals around the refined event time:
- `WITNESS`: $[-0.25, +0.5)\,\text{s}$ (96 samples at 128 Hz)
- `SENSOR`: $[-1.0, +2.0)\,\text{s}$ (384 samples)
- `DISPLACEMENT`: $[-2.0, +3.0)\,\text{s}$ (640 samples)

Each of the $8 \times 3 = 24$ event-channel pairs is cataloged with an explicit completeness status.

In [ ]:
channel_configs = {
    "WITNESS": {"pre_s": 0.25, "post_s": 0.5, "expected_samples": int(0.75 * 128)},
    "SENSOR": {"pre_s": 1.0, "post_s": 2.0, "expected_samples": int(3.0 * 128)},
    "DISPLACEMENT": {"pre_s": 2.0, "post_s": 3.0, "expected_samples": int(5.0 * 128)},
}

event_channel_records = []
aligned_arrays = {ch: [] for ch in channel_configs}
aligned_ids = {ch: [] for ch in channel_configs}

for _, row in refined_df.iterrows():
    evt_id = row["event_id"]
    t_center = row["refined_gps_s"]
    is_no_peak = (row["refine_status"] == "no_peak")

    for ch, cfg in channel_configs.items():
        ts = ts_dict[ch]
        req_start = t_center - cfg["pre_s"]
        req_end = t_center + cfg["post_s"]

        act_start = max(req_start, ts.span[0])
        act_end = min(req_end, ts.span[1])

        status = "complete"
        if is_no_peak:
            status = "no_peak"
        elif req_start < ts.span[0] or req_end > ts.span[1]:
            status = "partial" if (act_end > act_start) else "outside_data"

        n_samples = 0
        peak_val = np.nan
        rms_val = np.nan

        if act_end > act_start:
            sub = ts.crop(act_start * u.s, act_end * u.s)
            val = sub.value
            n_samples = len(val)
            if np.any(np.isnan(val)):
                status = "gap"
            else:
                peak_val = float(np.max(val))
                rms_val = float(np.sqrt(np.mean(val**2)))
                if status == "complete" and n_samples == cfg["expected_samples"]:
                    aligned_arrays[ch].append(val)
                    aligned_ids[ch].append(evt_id)

        event_channel_records.append({
            "event_id": evt_id,
            "channel": ch,
            "requested_start_s": float(req_start),
            "requested_end_s": float(req_end),
            "actual_start_s": float(act_start),
            "actual_end_s": float(act_end),
            "n_samples": int(n_samples),
            "peak": float(peak_val) if not np.isnan(peak_val) else None,
            "rms": float(rms_val) if not np.isnan(rms_val) else None,
            "unit": str(ts.unit),
            "status": status
        })

event_channels_df = pd.DataFrame(event_channel_records)
event_channels_df.to_csv(output_dir / "tables/event_channels.csv", index=False)
print("Event channels table saved to tables/event_channels.csv (24 rows)")

# Save aligned arrays
for ch, arrs in aligned_arrays.items():
    if arrs:
        np.savez(output_dir / f"tables/aligned_{ch.lower()}.npz",
                 events=np.array(aligned_ids[ch]),
                 data=np.array(arrs),
                 sample_rate=128.0)

## Diagnostic Plots and Waveform Stacking

In [ ]:
# Plot 1: Timing refinement
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(refined_df["event_id"], refined_df["offset_s"] * 1000, "o-", color="tab:blue", label="Refinement Offset")
ax.axhline(0, color="k", ls="--", alpha=0.5)
ax.set_ylabel("Offset [ms]")
ax.set_xlabel("Event ID")
ax.set_title("Catalog vs Refined Peak Timestamp Offset")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
fig.savefig(output_dir / "figures/catalog_refinement.png", dpi=150)
plt.close(fig)

# Plot 2: Aligned waveforms
fig, axs = plt.subplots(3, 1, figsize=(9, 8), sharex=False)
for idx, (ch, cfg) in enumerate(channel_configs.items()):
    arrs = aligned_arrays[ch]
    if arrs:
        rel_t = np.arange(cfg["expected_samples"]) / 128.0 - cfg["pre_s"]
        for arr in arrs:
            axs[idx].plot(rel_t, arr, alpha=0.5)
        mean_arr = np.mean(arrs, axis=0)
        axs[idx].plot(rel_t, mean_arr, color="k", lw=2, label="Mean Response")
        axs[idx].set_ylabel(f"{ch} [{ts_dict[ch].unit}]")
        axs[idx].grid(True, alpha=0.3)
        axs[idx].legend(loc="upper right")
axs[2].set_xlabel("Time Relative to Refined Trigger [s]")
plt.tight_layout()
fig.savefig(output_dir / "figures/aligned_waveforms.png", dpi=150)
plt.close(fig)
print("Diagnostic figures generated")

## Quality Metrics and Checks

We evaluate the strict contract:
- Exactly 8 catalog events and 24 event-channel rows.
- Refinement accuracy: internal pulse error $\le 1/128\,\text{s}$.
- Channel lag recovery: 4 samples sensor lag, 8 samples displacement lag.
- Status preservation for boundaries and gaps.

In [ ]:
# Check 1: ID preservation
row_count_ok = bool((len(catalog_df) == 8) and (len(event_channels_df) == 24))

# Check 2: Refinement accuracy on internal events (E01, E02, E04, E06)
internal_ids = ["E01", "E02", "E04", "E06"]
ref_errors = []
for eid in internal_ids:
    row_ref = refined_df[refined_df["event_id"] == eid].iloc[0]
    row_cat = catalog_df[catalog_df["event_id"] == eid].iloc[0]
    error = abs(row_ref["refined_gps_s"] - row_cat["design_gps_s"])
    ref_errors.append(error)
ref_acc_ok = bool((max(ref_errors) <= (1.0 / 128.0)) and (refined_df[refined_df["event_id"] == "E05"]["refine_status"].iloc[0] == "no_peak"))

# Check 3: Lag recovery
s_lags = []
d_lags = []
for eid in internal_ids:
    w_row = refined_df[refined_df["event_id"] == eid].iloc[0]
    s_sub = ts_dict["SENSOR"].crop((w_row["refined_gps_s"] - 0.1)*u.s, (w_row["refined_gps_s"] + 0.1)*u.s)
    d_sub = ts_dict["DISPLACEMENT"].crop((w_row["refined_gps_s"] - 0.1)*u.s, (w_row["refined_gps_s"] + 0.1)*u.s)
    s_peak_t = s_sub.times.value[np.argmax(s_sub.value)]
    d_peak_t = d_sub.times.value[np.argmax(d_sub.value)]
    s_lags.append(round((s_peak_t - w_row["refined_gps_s"]) * 128))
    d_lags.append(round((d_peak_t - w_row["refined_gps_s"]) * 128))

lag_ok = bool((np.median(s_lags) == 4) and (np.median(d_lags) == 8))

# Check 4: Edge & gap status preservation
e00_w = event_channels_df[(event_channels_df["event_id"] == "E00") & (event_channels_df["channel"] == "DISPLACEMENT")].iloc[0]
e03_s = event_channels_df[(event_channels_df["event_id"] == "E03") & (event_channels_df["channel"] == "SENSOR")].iloc[0]
status_ok = bool((e00_w["status"] == "partial") and (e03_s["status"] == "gap"))

metrics = {
    "status": "passed" if (row_count_ok and ref_acc_ok and lag_ok and status_ok) else "failed",
    "checks": {
        "event_id_preservation": {"passed": row_count_ok, "catalog_rows": len(catalog_df), "channel_rows": len(event_channels_df)},
        "event_refinement_accuracy": {"passed": ref_acc_ok, "max_error_s": float(max(ref_errors))},
        "event_lag_recovery": {"passed": lag_ok, "sensor_lag_samples": float(np.median(s_lags)), "disp_lag_samples": float(np.median(d_lags))},
        "event_edge_and_gap_status": {"passed": status_ok}
    }
}

with open(output_dir / "validation-metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

settings = {
    "tutorial_id": "T2",
    "fs_hz": 128.0,
    "events_count": 8,
    "channels": ["WITNESS", "SENSOR", "DISPLACEMENT"],
    "channel_windows_s": {ch: [cfg["pre_s"], cfg["post_s"]] for ch, cfg in channel_configs.items()}
}
with open(output_dir / "analysis-settings.json", "w", encoding="utf-8") as f:
    json.dump(settings, f, indent=2)

print("Validation metrics:")
print(json.dumps(metrics, indent=2))
assert metrics["status"] == "passed", "T2 verification checks failed!"